# 🫁 Pneumonia Detection - Deep Learning Project
## B.Tech Final Year - Complete Training & Evaluation
### 5 Models: Custom CNN, VGG16, ResNet50, DenseNet121, Vision Transformer

**Project**: Pneumonia detection from chest X-ray images  
**Models**: 5 different architectures  
**Evaluation**: Train/Val/Test sets with comprehensive metrics  
**Data Augmentation**: 5-7x effective dataset size increase  
**System**: 8GB RAM optimized (sequential training)  

---

## SECTION 0: Setup & Dataset Preparation (Run Once)

In [ ]:
!pip install -q tensorflow keras numpy pandas matplotlib seaborn scikit-learn opencv-python Pillow imghdr

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing import image_dataset_from_directory

import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import imghdr
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    f1_score, accuracy_score, precision_score, recall_score
)

# Set random seeds
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("="*80)
print("PNEUMONIA DETECTION - DEEP LEARNING PROJECT")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("✓ All libraries loaded!")
print("="*80)

C:\Users\risha\AppData\Local\Temp\ipykernel_20260\1726843405.py:10: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


PNEUMONIA DETECTION - DEEP LEARNING PROJECT
TensorFlow version: 2.20.0
GPU Available: False
✓ All libraries loaded!


In [2]:

# Configure dataset paths
BASE_DIR = r"D:\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray"
# For Colab: BASE_DIR = "/content/dataset/chest_xray"

DATA_DIR_TRAIN = os.path.join(BASE_DIR, "train")
DATA_DIR_VAL = os.path.join(BASE_DIR, "val")
DATA_DIR_TEST = os.path.join(BASE_DIR, "test")

print("\n📂 DATASET PATHS:")
for name, path in [("Train", DATA_DIR_TRAIN), ("Val", DATA_DIR_VAL), ("Test", DATA_DIR_TEST)]:
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {name}: {path}")

# Configuration
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 16
EPOCHS = 50

# Create output directories
OUTPUT_DIR = "./results"
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")

for directory in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"\n⚙️ CONFIGURATION:")
print(f"  Image Size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Max Epochs: {EPOCHS}")


📂 DATASET PATHS:
  ✓ Train: D:\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\train
  ✓ Val: D:\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\val
  ✓ Test: D:\Major Project(Final-Year)\chest-xray-pneumonia-detection\dataset\chest_xray\test

⚙️ CONFIGURATION:
  Image Size: 224x224
  Batch Size: 16
  Max Epochs: 50


In [3]:
# Validate dataset
print("\nValidating dataset...")
image_extensions = [".png", ".jpg", ".jpeg"]
valid_images = 0

for filepath in Path(BASE_DIR).rglob("*"):
    if filepath.suffix.lower() in image_extensions:
        if imghdr.what(filepath):
            valid_images += 1

print(f"✓ Valid images: {valid_images}")

# Count class distribution
train_normal = len(list(Path(DATA_DIR_TRAIN).glob("NORMAL/*")))
train_pneumonia = len(list(Path(DATA_DIR_TRAIN).glob("PNEUMONIA/*")))
val_normal = len(list(Path(DATA_DIR_VAL).glob("NORMAL/*")))
val_pneumonia = len(list(Path(DATA_DIR_VAL).glob("PNEUMONIA/*")))
test_normal = len(list(Path(DATA_DIR_TEST).glob("NORMAL/*")))
test_pneumonia = len(list(Path(DATA_DIR_TEST).glob("PNEUMONIA/*")))

print(f"\n📊 Dataset Distribution:")
print(f"  Train: {train_normal + train_pneumonia} (Normal: {train_normal}, Pneumonia: {train_pneumonia})")
print(f"  Val: {val_normal + val_pneumonia} (Normal: {val_normal}, Pneumonia: {val_pneumonia})")
print(f"  Test: {test_normal + test_pneumonia} (Normal: {test_normal}, Pneumonia: {test_pneumonia})")
print(f"  Total: {train_normal + train_pneumonia + val_normal + val_pneumonia + test_normal + test_pneumonia}")


Validating dataset...
✓ Valid images: 5856

📊 Dataset Distribution:
  Train: 5218 (Normal: 1342, Pneumonia: 3876)
  Val: 18 (Normal: 9, Pneumonia: 9)
  Test: 624 (Normal: 234, Pneumonia: 390)
  Total: 5860


In [4]:
# Data augmentation setup
print("\nSetting up data augmentation...")

train_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.2, seed=SEED),
    layers.RandomZoom(0.2, seed=SEED),
    layers.RandomTranslation(0.1, 0.1, seed=SEED),
    layers.RandomBrightness(0.2, seed=SEED),
    layers.RandomContrast(0.2, seed=SEED),
    layers.GaussianNoise(0.01),
])

val_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.1, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
])

print("✓ Train augmentation: Heavy (5-7x effective size)")
print("✓ Val augmentation: Light")
print("✓ Test: No augmentation")


Setting up data augmentation...
✓ Train augmentation: Heavy (5-7x effective size)
✓ Val augmentation: Light
✓ Test: No augmentation


In [5]:
# ALL THE CODE YOU NEED - Copy and run in one cell

import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np

# Assuming you've already defined:
# DATA_DIR_TRAIN, DATA_DIR_VAL, DATA_DIR_TEST
# train_augmentation, val_augmentation
# IMG_HEIGHT, IMG_WIDTH, BATCH_SIZE, SEED

# Load datasets with correct order
train_ds_raw = image_dataset_from_directory(
    DATA_DIR_TRAIN, seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, label_mode='categorical'
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

train_ds = train_ds_raw.map(
    lambda x, y: (train_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_ds_raw = image_dataset_from_directory(
    DATA_DIR_VAL, seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, label_mode='categorical'
)

val_ds = val_ds_raw.map(
    lambda x, y: (val_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_ds = image_dataset_from_directory(
    DATA_DIR_TEST, seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
).prefetch(tf.data.AUTOTUNE)

# Extract labels
true_labels_test = np.array([np.argmax(y.numpy()) 
                              for _, y in test_ds for _ in range(len(y))])
true_labels_val = np.array([np.argmax(y.numpy()) 
                             for _, y in val_ds for _ in range(len(y))])
true_labels_train = np.array([np.argmax(y.numpy()) 
                               for _, y in train_ds for _ in range(len(y))])

# Verify everything loaded
print(f"✓ Classes: {class_names}")
print(f"✓ Train/Val/Test batches: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")
print(f"✓ All systems ready for training!")


Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.
✓ Classes: ['NORMAL', 'PNEUMONIA']
✓ Train/Val/Test batches: 326/1/39
✓ All systems ready for training!


## SECTION 1: Train Custom CNN Model

In [6]:
print("\n" + "="*80)
print("MODEL 1: CUSTOM CNN - TRAINING")
print("="*80)

import gc
gc.collect()
tf.keras.backend.clear_session()

custom_cnn_model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

custom_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

print(f"\n✓ Custom CNN Model")
print(f"  Parameters: {custom_cnn_model.count_params():,}")

callbacks_custom = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "custom_cnn_best.h5"),
                   monitor='val_accuracy', save_best_only=True, verbose=0)
]

print(f"\nStarting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}...\n")
start_time = datetime.now()

history_custom = custom_cnn_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_custom, verbose=1
)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\n✓ Training completed in {training_time:.2f} minutes")


MODEL 1: CUSTOM CNN - TRAINING


✓ Custom CNN Model
  Parameters: 323,362

Starting training at 2025-12-05 15:18:44...

Epoch 1/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7195 - auc: 0.7771 - loss: 0.7074 - precision: 0.7195 - recall: 0.7195

326/326 ━━━━━━━━━━━━━━━━━━━━ 1329s 4s/step - accuracy: 0.7467 - auc: 0.8191 - loss: 0.6158 - precision: 0.7467 - recall: 0.7467 - val_accuracy: 0.5000 - val_auc: 0.5625 - val_loss: 2.6534 - val_precision: 0.5000 - val_recall: 0.5000 - learning_rate: 0.0010
Epoch 2/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7957 - auc: 0.8766 - loss: 0.4760 - precision: 0.7957 - recall: 0.7957

326/326 ━━━━━━━━━━━━━━━━━━━━ 1083s 3s/step - accuracy: 0.8041 - auc: 0.8860 - loss: 0.4509 - precision: 0.8041 - recall: 0.8041 - val_accuracy: 0.5625 - val_auc: 0.5586 - val_loss: 0.9590 - val_precision: 0.5625 - val_recall: 0.5625 - learning_rate: 0.0010
Epoch 3/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1998s 6s/step - accuracy: 0.8221 - auc: 0.9061 - loss: 0.3954 - precision: 0.8221 - recall: 0.8221 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 4.2151 - val_precision: 0.5000 - val_recall: 0.5000 - learning_rate: 0.0010
Epoch 4/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1351s 4s/step - accuracy: 0.8368 - auc: 0.9220 - loss: 0.3553 - precision: 0.8368 - recall: 0.8368 - val_accuracy: 0.5625 - val_auc: 0.5977 - val_loss: 2.6197 - val_precision: 0.5625 - val_recall: 0.5625 - learning_rate: 0.0010
Epoch 5/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1295s 4s/step - accuracy: 0.8581 - auc: 0.9337 - loss: 0.3303 - precision: 0.8581 - recall: 0.8581 - val_accuracy: 0.5000 - val_auc: 0.4648 - val_loss: 1.2181 - val_pr

326/326 ━━━━━━━━━━━━━━━━━━━━ 1130s 3s/step - accuracy: 0.9064 - auc: 0.9685 - loss: 0.2277 - precision: 0.9064 - recall: 0.9064 - val_accuracy: 0.7500 - val_auc: 0.7500 - val_loss: 0.6953 - val_precision: 0.7500 - val_recall: 0.7500 - learning_rate: 5.0000e-04
Epoch 14/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1133s 3s/step - accuracy: 0.9105 - auc: 0.9693 - loss: 0.2240 - precision: 0.9105 - recall: 0.9105 - val_accuracy: 0.6250 - val_auc: 0.7266 - val_loss: 1.2467 - val_precision: 0.6250 - val_recall: 0.6250 - learning_rate: 5.0000e-04
Epoch 15/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1087s 3s/step - accuracy: 0.9122 - auc: 0.9721 - loss: 0.2133 - precision: 0.9122 - recall: 0.9122 - val_accuracy: 0.5000 - val_auc: 0.6250 - val_loss: 2.7093 - val_precision: 0.5000 - val_recall: 0.5000 - learning_rate: 5.0000e-04
Epoch 16/50
326/326 ━━━━━━━━━━━━━━━━━━━━ 1089s 3s/step - accuracy: 0.9185 - auc: 0.9747 - loss: 0.2037 - precision: 0.9185 - recall: 0.9185 - val_accuracy: 0.5625 - val_auc: 0.5898 - val_loss: 

KeyboardInterrupt: 

In [ ]:
# Evaluate Custom CNN
print("\n" + "="*80)
print("📊 CUSTOM CNN: EVALUATION")
print("="*80)

# Training Set
print("\n" + "-"*80)
print("TRAINING SET EVALUATION")
print("-"*80)
predictions_train = custom_cnn_model.predict(train_ds, verbose=0)
predicted_labels_train = np.argmax(predictions_train, axis=1)

train_acc = accuracy_score(true_labels_train, predicted_labels_train)
train_prec = precision_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_rec = recall_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_f1 = f1_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_auc = roc_auc_score(true_labels_train, predictions_train, multi_class='ovr', average='weighted')

print(f"Accuracy:  {train_acc:.4f} | Precision: {train_prec:.4f} | Recall: {train_rec:.4f}")
print(f"F1-Score:  {train_f1:.4f} | ROC-AUC:   {train_auc:.4f}")

# Validation Set
print("\n" + "-"*80)
print("VALIDATION SET EVALUATION")
print("-"*80)
predictions_val = custom_cnn_model.predict(val_ds, verbose=0)
predicted_labels_val = np.argmax(predictions_val, axis=1)

val_acc = accuracy_score(true_labels_val, predicted_labels_val)
val_prec = precision_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_rec = recall_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_f1 = f1_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_auc = roc_auc_score(true_labels_val, predictions_val, multi_class='ovr', average='weighted')

print(f"Accuracy:  {val_acc:.4f} | Precision: {val_prec:.4f} | Recall: {val_rec:.4f}")
print(f"F1-Score:  {val_f1:.4f} | ROC-AUC:   {val_auc:.4f}")

# Test Set
print("\n" + "-"*80)
print("TEST SET EVALUATION (Original Images)")
print("-"*80)
predictions_test = custom_cnn_model.predict(test_ds, verbose=0)
predicted_labels_test = np.argmax(predictions_test, axis=1)

test_acc = accuracy_score(true_labels_test, predicted_labels_test)
test_prec = precision_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_rec = recall_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_f1 = f1_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_auc = roc_auc_score(true_labels_test, predictions_test, multi_class='ovr', average='weighted')

print(f"Accuracy:  {test_acc:.4f} | Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")
print(f"F1-Score:  {test_f1:.4f} | ROC-AUC:   {test_auc:.4f}")

print("\n" + "-"*80)
print("CLASSIFICATION REPORT (Test Set)")
print("-"*80)
print(classification_report(true_labels_test, predicted_labels_test, target_names=class_names, digits=4))

custom_cnn_results = {
    'train': {'accuracy': train_acc, 'precision': train_prec, 'recall': train_rec, 'f1': train_f1, 'auc': train_auc},
    'val': {'accuracy': val_acc, 'precision': val_prec, 'recall': val_rec, 'f1': val_f1, 'auc': val_auc},
    'test': {'accuracy': test_acc, 'precision': test_prec, 'recall': test_rec, 'f1': test_f1, 'auc': test_auc}
}

In [ ]:
# Plot confusion matrices
cm_train = confusion_matrix(true_labels_train, predicted_labels_train)
cm_val = confusion_matrix(true_labels_val, predicted_labels_val)
cm_test = confusion_matrix(true_labels_test, predicted_labels_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Custom CNN - Confusion Matrices', fontsize=14, fontweight='bold')

for idx, (cm, title) in enumerate([(cm_train, 'Train'), (cm_val, 'Val'), (cm_test, 'Test')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_custom_cnn_confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved")

In [ ]:
# Training history plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Custom CNN - Training History', fontsize=14, fontweight='bold')

axes[0, 0].plot(history_custom.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_custom.history['val_accuracy'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_custom.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_custom.history['val_loss'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history_custom.history['precision'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 0].plot(history_custom.history['val_precision'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_custom.history['recall'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 1].plot(history_custom.history['val_recall'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_custom_cnn_training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training history plots saved")
print("\n" + "="*80)
print("🎉 CUSTOM CNN COMPLETE")
print("="*80)
print("\n⚠️  Restart kernel before next model: Kernel → Restart Kernel")

## SECTION 2: Train VGG16 Model

In [ ]:
print("\n" + "="*80)
print("MODEL 2: VGG16 - TRAINING")
print("="*80)

import gc
gc.collect()
tf.keras.backend.clear_session()

base_model_vgg = applications.VGG16(weights='imagenet', include_top=False,
                                    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model_vgg.trainable = False

vgg16_model = models.Sequential([
    layers.Rescaling(1./255),
    base_model_vgg,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

vgg16_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

print(f"✓ VGG16 Model\n  Parameters: {vgg16_model.count_params():,}")

callbacks_vgg = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "vgg16_best.h5"),
                   monitor='val_accuracy', save_best_only=True, verbose=0)
]

print(f"\nStarting training...\n")
start_time = datetime.now()

history_vgg = vgg16_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_vgg, verbose=1
)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\n✓ Training completed in {training_time:.2f} minutes")

In [ ]:
# Evaluate VGG16
print("\n" + "="*80)
print("📊 VGG16: EVALUATION")
print("="*80)

print("\n" + "-"*80 + "\nTRAINING SET EVALUATION\n" + "-"*80)
predictions_train = vgg16_model.predict(train_ds, verbose=0)
predicted_labels_train = np.argmax(predictions_train, axis=1)
train_acc_vgg = accuracy_score(true_labels_train, predicted_labels_train)
train_prec_vgg = precision_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_rec_vgg = recall_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_f1_vgg = f1_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_auc_vgg = roc_auc_score(true_labels_train, predictions_train, multi_class='ovr', average='weighted')
print(f"Accuracy: {train_acc_vgg:.4f} | Precision: {train_prec_vgg:.4f} | Recall: {train_rec_vgg:.4f}")
print(f"F1: {train_f1_vgg:.4f} | AUC: {train_auc_vgg:.4f}")

print("\n" + "-"*80 + "\nVALIDATION SET EVALUATION\n" + "-"*80)
predictions_val = vgg16_model.predict(val_ds, verbose=0)
predicted_labels_val = np.argmax(predictions_val, axis=1)
val_acc_vgg = accuracy_score(true_labels_val, predicted_labels_val)
val_prec_vgg = precision_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_rec_vgg = recall_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_f1_vgg = f1_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_auc_vgg = roc_auc_score(true_labels_val, predictions_val, multi_class='ovr', average='weighted')
print(f"Accuracy: {val_acc_vgg:.4f} | Precision: {val_prec_vgg:.4f} | Recall: {val_rec_vgg:.4f}")
print(f"F1: {val_f1_vgg:.4f} | AUC: {val_auc_vgg:.4f}")

print("\n" + "-"*80 + "\nTEST SET EVALUATION\n" + "-"*80)
predictions_test = vgg16_model.predict(test_ds, verbose=0)
predicted_labels_test = np.argmax(predictions_test, axis=1)
test_acc_vgg = accuracy_score(true_labels_test, predicted_labels_test)
test_prec_vgg = precision_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_rec_vgg = recall_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_f1_vgg = f1_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_auc_vgg = roc_auc_score(true_labels_test, predictions_test, multi_class='ovr', average='weighted')
print(f"Accuracy: {test_acc_vgg:.4f} | Precision: {test_prec_vgg:.4f} | Recall: {test_rec_vgg:.4f}")
print(f"F1: {test_f1_vgg:.4f} | AUC: {test_auc_vgg:.4f}")

print("\n" + "-"*80 + "\nCLASSIFICATION REPORT\n" + "-"*80)
print(classification_report(true_labels_test, predicted_labels_test, target_names=class_names, digits=4))

vgg16_results = {
    'train': {'accuracy': train_acc_vgg, 'precision': train_prec_vgg, 'recall': train_rec_vgg, 'f1': train_f1_vgg, 'auc': train_auc_vgg},
    'val': {'accuracy': val_acc_vgg, 'precision': val_prec_vgg, 'recall': val_rec_vgg, 'f1': val_f1_vgg, 'auc': val_auc_vgg},
    'test': {'accuracy': test_acc_vgg, 'precision': test_prec_vgg, 'recall': test_rec_vgg, 'f1': test_f1_vgg, 'auc': test_auc_vgg}
}

In [ ]:
# Plot visualizations
cm_train = confusion_matrix(true_labels_train, predicted_labels_train)
cm_val = confusion_matrix(true_labels_val, predicted_labels_val)
cm_test = confusion_matrix(true_labels_test, predicted_labels_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VGG16 - Confusion Matrices', fontsize=14, fontweight='bold')
for idx, (cm, title) in enumerate([(cm_train, 'Train'), (cm_val, 'Val'), (cm_test, 'Test')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[idx],
               xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "02_vgg16_confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('VGG16 - Training History', fontsize=14, fontweight='bold')
axes[0, 0].plot(history_vgg.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_vgg.history['val_accuracy'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].plot(history_vgg.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_vgg.history['val_loss'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(history_vgg.history['precision'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 0].plot(history_vgg.history['val_precision'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].plot(history_vgg.history['recall'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 1].plot(history_vgg.history['val_recall'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "02_vgg16_training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("🎉 VGG16 COMPLETE")
print("="*80)
print("\n⚠️  Restart kernel before next model")

## SECTION 3: Train ResNet50 Model

In [ ]:
print("\n" + "="*80)
print("MODEL 3: ResNet50 - TRAINING")
print("="*80)

import gc
gc.collect()
tf.keras.backend.clear_session()

base_model_res = applications.ResNet50(weights='imagenet', include_top=False,
                                       input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model_res.trainable = False

resnet50_model = models.Sequential([
    layers.Rescaling(1./255),
    base_model_res,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

print(f"✓ ResNet50 Model\n  Parameters: {resnet50_model.count_params():,}")

callbacks_res = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "resnet50_best.h5"),
                   monitor='val_accuracy', save_best_only=True, verbose=0)
]

print(f"\nStarting training...\n")
start_time = datetime.now()

history_res = resnet50_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_res, verbose=1
)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\n✓ Training completed in {training_time:.2f} minutes")

In [ ]:
# Evaluate ResNet50
print("\n" + "="*80)
print("📊 ResNet50: EVALUATION")
print("="*80)

print("\n" + "-"*80 + "\nTRAINING SET EVALUATION\n" + "-"*80)
predictions_train = resnet50_model.predict(train_ds, verbose=0)
predicted_labels_train = np.argmax(predictions_train, axis=1)
train_acc_res = accuracy_score(true_labels_train, predicted_labels_train)
train_prec_res = precision_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_rec_res = recall_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_f1_res = f1_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_auc_res = roc_auc_score(true_labels_train, predictions_train, multi_class='ovr', average='weighted')
print(f"Accuracy: {train_acc_res:.4f} | Precision: {train_prec_res:.4f} | Recall: {train_rec_res:.4f}")
print(f"F1: {train_f1_res:.4f} | AUC: {train_auc_res:.4f}")

print("\n" + "-"*80 + "\nVALIDATION SET EVALUATION\n" + "-"*80)
predictions_val = resnet50_model.predict(val_ds, verbose=0)
predicted_labels_val = np.argmax(predictions_val, axis=1)
val_acc_res = accuracy_score(true_labels_val, predicted_labels_val)
val_prec_res = precision_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_rec_res = recall_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_f1_res = f1_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_auc_res = roc_auc_score(true_labels_val, predictions_val, multi_class='ovr', average='weighted')
print(f"Accuracy: {val_acc_res:.4f} | Precision: {val_prec_res:.4f} | Recall: {val_rec_res:.4f}")
print(f"F1: {val_f1_res:.4f} | AUC: {val_auc_res:.4f}")

print("\n" + "-"*80 + "\nTEST SET EVALUATION\n" + "-"*80)
predictions_test = resnet50_model.predict(test_ds, verbose=0)
predicted_labels_test = np.argmax(predictions_test, axis=1)
test_acc_res = accuracy_score(true_labels_test, predicted_labels_test)
test_prec_res = precision_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_rec_res = recall_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_f1_res = f1_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_auc_res = roc_auc_score(true_labels_test, predictions_test, multi_class='ovr', average='weighted')
print(f"Accuracy: {test_acc_res:.4f} | Precision: {test_prec_res:.4f} | Recall: {test_rec_res:.4f}")
print(f"F1: {test_f1_res:.4f} | AUC: {test_auc_res:.4f}")

print("\n" + "-"*80 + "\nCLASSIFICATION REPORT\n" + "-"*80)
print(classification_report(true_labels_test, predicted_labels_test, target_names=class_names, digits=4))

resnet50_results = {
    'train': {'accuracy': train_acc_res, 'precision': train_prec_res, 'recall': train_rec_res, 'f1': train_f1_res, 'auc': train_auc_res},
    'val': {'accuracy': val_acc_res, 'precision': val_prec_res, 'recall': val_rec_res, 'f1': val_f1_res, 'auc': val_auc_res},
    'test': {'accuracy': test_acc_res, 'precision': test_prec_res, 'recall': test_rec_res, 'f1': test_f1_res, 'auc': test_auc_res}
}

In [ ]:
# Plot visualizations
cm_train = confusion_matrix(true_labels_train, predicted_labels_train)
cm_val = confusion_matrix(true_labels_val, predicted_labels_val)
cm_test = confusion_matrix(true_labels_test, predicted_labels_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ResNet50 - Confusion Matrices', fontsize=14, fontweight='bold')
for idx, (cm, title) in enumerate([(cm_train, 'Train'), (cm_val, 'Val'), (cm_test, 'Test')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[idx],
               xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "03_resnet50_confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('ResNet50 - Training History', fontsize=14, fontweight='bold')
axes[0, 0].plot(history_res.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_res.history['val_accuracy'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].plot(history_res.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_res.history['val_loss'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(history_res.history['precision'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 0].plot(history_res.history['val_precision'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].plot(history_res.history['recall'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 1].plot(history_res.history['val_recall'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "03_resnet50_training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("🎉 ResNet50 COMPLETE")
print("="*80)
print("\n⚠️  Restart kernel before next model")

## SECTION 4: Train DenseNet121 Model

In [ ]:
print("\n" + "="*80)
print("MODEL 4: DenseNet121 - TRAINING")
print("="*80)

import gc
gc.collect()
tf.keras.backend.clear_session()

base_model_dense = applications.DenseNet121(weights='imagenet', include_top=False,
                                            input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model_dense.trainable = False

densenet_model = models.Sequential([
    layers.Rescaling(1./255),
    base_model_dense,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

print(f"✓ DenseNet121 Model\n  Parameters: {densenet_model.count_params():,}")

callbacks_dense = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "densenet_best.h5"),
                   monitor='val_accuracy', save_best_only=True, verbose=0)
]

print(f"\nStarting training...\n")
start_time = datetime.now()

history_dense = densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_dense, verbose=1
)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\n✓ Training completed in {training_time:.2f} minutes")

In [ ]:
# Evaluate DenseNet121
print("\n" + "="*80)
print("📊 DenseNet121: EVALUATION")
print("="*80)

print("\n" + "-"*80 + "\nTRAINING SET EVALUATION\n" + "-"*80)
predictions_train = densenet_model.predict(train_ds, verbose=0)
predicted_labels_train = np.argmax(predictions_train, axis=1)
train_acc_dense = accuracy_score(true_labels_train, predicted_labels_train)
train_prec_dense = precision_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_rec_dense = recall_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_f1_dense = f1_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_auc_dense = roc_auc_score(true_labels_train, predictions_train, multi_class='ovr', average='weighted')
print(f"Accuracy: {train_acc_dense:.4f} | Precision: {train_prec_dense:.4f} | Recall: {train_rec_dense:.4f}")
print(f"F1: {train_f1_dense:.4f} | AUC: {train_auc_dense:.4f}")

print("\n" + "-"*80 + "\nVALIDATION SET EVALUATION\n" + "-"*80)
predictions_val = densenet_model.predict(val_ds, verbose=0)
predicted_labels_val = np.argmax(predictions_val, axis=1)
val_acc_dense = accuracy_score(true_labels_val, predicted_labels_val)
val_prec_dense = precision_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_rec_dense = recall_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_f1_dense = f1_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_auc_dense = roc_auc_score(true_labels_val, predictions_val, multi_class='ovr', average='weighted')
print(f"Accuracy: {val_acc_dense:.4f} | Precision: {val_prec_dense:.4f} | Recall: {val_rec_dense:.4f}")
print(f"F1: {val_f1_dense:.4f} | AUC: {val_auc_dense:.4f}")

print("\n" + "-"*80 + "\nTEST SET EVALUATION\n" + "-"*80)
predictions_test = densenet_model.predict(test_ds, verbose=0)
predicted_labels_test = np.argmax(predictions_test, axis=1)
test_acc_dense = accuracy_score(true_labels_test, predicted_labels_test)
test_prec_dense = precision_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_rec_dense = recall_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_f1_dense = f1_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_auc_dense = roc_auc_score(true_labels_test, predictions_test, multi_class='ovr', average='weighted')
print(f"Accuracy: {test_acc_dense:.4f} | Precision: {test_prec_dense:.4f} | Recall: {test_rec_dense:.4f}")
print(f"F1: {test_f1_dense:.4f} | AUC: {test_auc_dense:.4f}")

print("\n" + "-"*80 + "\nCLASSIFICATION REPORT\n" + "-"*80)
print(classification_report(true_labels_test, predicted_labels_test, target_names=class_names, digits=4))

densenet_results = {
    'train': {'accuracy': train_acc_dense, 'precision': train_prec_dense, 'recall': train_rec_dense, 'f1': train_f1_dense, 'auc': train_auc_dense},
    'val': {'accuracy': val_acc_dense, 'precision': val_prec_dense, 'recall': val_rec_dense, 'f1': val_f1_dense, 'auc': val_auc_dense},
    'test': {'accuracy': test_acc_dense, 'precision': test_prec_dense, 'recall': test_rec_dense, 'f1': test_f1_dense, 'auc': test_auc_dense}
}

In [ ]:
# Plot visualizations
cm_train = confusion_matrix(true_labels_train, predicted_labels_train)
cm_val = confusion_matrix(true_labels_val, predicted_labels_val)
cm_test = confusion_matrix(true_labels_test, predicted_labels_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DenseNet121 - Confusion Matrices', fontsize=14, fontweight='bold')
for idx, (cm, title) in enumerate([(cm_train, 'Train'), (cm_val, 'Val'), (cm_test, 'Test')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[idx],
               xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "04_densenet_confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('DenseNet121 - Training History', fontsize=14, fontweight='bold')
axes[0, 0].plot(history_dense.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_dense.history['val_accuracy'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].plot(history_dense.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_dense.history['val_loss'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(history_dense.history['precision'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 0].plot(history_dense.history['val_precision'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].plot(history_dense.history['recall'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 1].plot(history_dense.history['val_recall'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "04_densenet_training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("🎉 DenseNet121 COMPLETE")
print("="*80)
print("\n⚠️  Restart kernel before next model")

## SECTION 5: Train Vision Transformer Model

In [ ]:
print("\n" + "="*80)
print("MODEL 5: VISION TRANSFORMER - TRAINING")
print("="*80)

import gc
gc.collect()
tf.keras.backend.clear_session()

def create_vision_transformer():
    patch_size = 16
    num_patches = (IMG_HEIGHT // patch_size) ** 2
    patch_dim = 3 * patch_size * patch_size
    transformer_dim = 512
    
    inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = layers.Rescaling(1./255)(inputs)
    patches = layers.Reshape((IMG_HEIGHT // patch_size, patch_size, IMG_WIDTH // patch_size, patch_size, 3))(x)
    patches = layers.Permute((1, 3, 2, 4, 5))(patches)
    patches = layers.Reshape((num_patches, patch_dim))(patches)
    x = layers.Dense(transformer_dim, activation='relu')(patches)
    
    for _ in range(4):
        attention = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
        x = layers.Add()([x, attention])
        x = layers.LayerNormalization()(x)
        ff = layers.Dense(2048, activation='relu')(x)
        ff = layers.Dense(transformer_dim)(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization()(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

vit_model = create_vision_transformer()

vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

print(f"✓ Vision Transformer Model\n  Parameters: {vit_model.count_params():,}")

callbacks_vit = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=os.path.join(MODELS_DIR, "vision_transformer_best.h5"),
                   monitor='val_accuracy', save_best_only=True, verbose=0)
]

print(f"\nStarting training...\n")
start_time = datetime.now()

history_vit = vit_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_vit, verbose=1
)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\n✓ Training completed in {training_time:.2f} minutes")

In [ ]:
# Evaluate Vision Transformer
print("\n" + "="*80)
print("📊 VISION TRANSFORMER: EVALUATION")
print("="*80)

print("\n" + "-"*80 + "\nTRAINING SET EVALUATION\n" + "-"*80)
predictions_train = vit_model.predict(train_ds, verbose=0)
predicted_labels_train = np.argmax(predictions_train, axis=1)
train_acc_vit = accuracy_score(true_labels_train, predicted_labels_train)
train_prec_vit = precision_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_rec_vit = recall_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_f1_vit = f1_score(true_labels_train, predicted_labels_train, average='weighted', zero_division=0)
train_auc_vit = roc_auc_score(true_labels_train, predictions_train, multi_class='ovr', average='weighted')
print(f"Accuracy: {train_acc_vit:.4f} | Precision: {train_prec_vit:.4f} | Recall: {train_rec_vit:.4f}")
print(f"F1: {train_f1_vit:.4f} | AUC: {train_auc_vit:.4f}")

print("\n" + "-"*80 + "\nVALIDATION SET EVALUATION\n" + "-"*80)
predictions_val = vit_model.predict(val_ds, verbose=0)
predicted_labels_val = np.argmax(predictions_val, axis=1)
val_acc_vit = accuracy_score(true_labels_val, predicted_labels_val)
val_prec_vit = precision_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_rec_vit = recall_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_f1_vit = f1_score(true_labels_val, predicted_labels_val, average='weighted', zero_division=0)
val_auc_vit = roc_auc_score(true_labels_val, predictions_val, multi_class='ovr', average='weighted')
print(f"Accuracy: {val_acc_vit:.4f} | Precision: {val_prec_vit:.4f} | Recall: {val_rec_vit:.4f}")
print(f"F1: {val_f1_vit:.4f} | AUC: {val_auc_vit:.4f}")

print("\n" + "-"*80 + "\nTEST SET EVALUATION\n" + "-"*80)
predictions_test = vit_model.predict(test_ds, verbose=0)
predicted_labels_test = np.argmax(predictions_test, axis=1)
test_acc_vit = accuracy_score(true_labels_test, predicted_labels_test)
test_prec_vit = precision_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_rec_vit = recall_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_f1_vit = f1_score(true_labels_test, predicted_labels_test, average='weighted', zero_division=0)
test_auc_vit = roc_auc_score(true_labels_test, predictions_test, multi_class='ovr', average='weighted')
print(f"Accuracy: {test_acc_vit:.4f} | Precision: {test_prec_vit:.4f} | Recall: {test_rec_vit:.4f}")
print(f"F1: {test_f1_vit:.4f} | AUC: {test_auc_vit:.4f}")

print("\n" + "-"*80 + "\nCLASSIFICATION REPORT\n" + "-"*80)
print(classification_report(true_labels_test, predicted_labels_test, target_names=class_names, digits=4))

vit_results = {
    'train': {'accuracy': train_acc_vit, 'precision': train_prec_vit, 'recall': train_rec_vit, 'f1': train_f1_vit, 'auc': train_auc_vit},
    'val': {'accuracy': val_acc_vit, 'precision': val_prec_vit, 'recall': val_rec_vit, 'f1': val_f1_vit, 'auc': val_auc_vit},
    'test': {'accuracy': test_acc_vit, 'precision': test_prec_vit, 'recall': test_rec_vit, 'f1': test_f1_vit, 'auc': test_auc_vit}
}

In [ ]:
# Plot visualizations
cm_train = confusion_matrix(true_labels_train, predicted_labels_train)
cm_val = confusion_matrix(true_labels_val, predicted_labels_val)
cm_test = confusion_matrix(true_labels_test, predicted_labels_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Vision Transformer - Confusion Matrices', fontsize=14, fontweight='bold')
for idx, (cm, title) in enumerate([(cm_train, 'Train'), (cm_val, 'Val'), (cm_test, 'Test')]):
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', ax=axes[idx],
               xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "05_vit_confusion_matrices.png"), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Vision Transformer - Training History', fontsize=14, fontweight='bold')
axes[0, 0].plot(history_vit.history['accuracy'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_vit.history['val_accuracy'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].plot(history_vit.history['loss'], label='Train', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_vit.history['val_loss'], label='Val', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(history_vit.history['precision'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 0].plot(history_vit.history['val_precision'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].plot(history_vit.history['recall'], label='Train', linewidth=2, marker='o', markersize=4)
axes[1, 1].plot(history_vit.history['val_recall'], label='Val', linewidth=2, marker='s', markersize=4)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "05_vit_training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("🎉 VISION TRANSFORMER COMPLETE")
print("✅ ALL 5 MODELS TRAINED SUCCESSFULLY!")
print("="*80)

## FINAL COMPARISON: All Models

In [ ]:
print("\n" + "="*100)
print("📊 FINAL COMPARISON: ALL 5 MODELS")
print("="*100)

# Create comparison dataframes
test_results_df = pd.DataFrame({
    'Custom CNN': custom_cnn_results['test'],
    'VGG16': vgg16_results['test'],
    'ResNet50': resnet50_results['test'],
    'DenseNet121': densenet_results['test'],
    'Vision Transformer': vit_results['test']
}).T

val_results_df = pd.DataFrame({
    'Custom CNN': custom_cnn_results['val'],
    'VGG16': vgg16_results['val'],
    'ResNet50': resnet50_results['val'],
    'DenseNet121': densenet_results['val'],
    'Vision Transformer': vit_results['val']
}).T

train_results_df = pd.DataFrame({
    'Custom CNN': custom_cnn_results['train'],
    'VGG16': vgg16_results['train'],
    'ResNet50': resnet50_results['train'],
    'DenseNet121': densenet_results['train'],
    'Vision Transformer': vit_results['train']
}).T

# Display results
print("\n" + "-"*100)
print("TRAINING SET RESULTS")
print("-"*100)
print(train_results_df.round(4).to_string())

print("\n" + "-"*100)
print("VALIDATION SET RESULTS")
print("-"*100)
print(val_results_df.round(4).to_string())

print("\n" + "-"*100)
print("TEST SET RESULTS")
print("-"*100)
print(test_results_df.round(4).to_string())

# Save to CSV
train_results_df.round(4).to_csv(os.path.join(METRICS_DIR, "train_results.csv"))
val_results_df.round(4).to_csv(os.path.join(METRICS_DIR, "val_results.csv"))
test_results_df.round(4).to_csv(os.path.join(METRICS_DIR, "test_results.csv"))

print("\n✓ Results saved to CSV files")

# Rankings
print("\n" + "="*100)
print("🏆 MODEL RANKINGS (by Test Accuracy)")
print("="*100)
rankings = test_results_df['accuracy'].sort_values(ascending=False)
for idx, (model, acc) in enumerate(rankings.items(), 1):
    print(f"  {idx}. {model}: {acc:.4f}")

print("\n" + "="*100)

In [ ]:
# Comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance Comparison (Test Set)', fontsize=16, fontweight='bold')

models_list = list(test_results_df.index)

# Accuracy
accuracies = test_results_df['accuracy'].values
axes[0, 0].bar(models_list, accuracies, color='skyblue', edgecolor='navy')
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Precision
precisions = test_results_df['precision'].values
axes[0, 1].bar(models_list, precisions, color='lightcoral', edgecolor='darkred')
axes[0, 1].set_title('Precision', fontweight='bold')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(precisions):
    axes[0, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Recall
recalls = test_results_df['recall'].values
axes[0, 2].bar(models_list, recalls, color='lightgreen', edgecolor='darkgreen')
axes[0, 2].set_title('Recall', fontweight='bold')
axes[0, 2].set_ylabel('Recall')
axes[0, 2].tick_params(axis='x', rotation=45)
for i, v in enumerate(recalls):
    axes[0, 2].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# F1
f1_scores = test_results_df['f1'].values
axes[1, 0].bar(models_list, f1_scores, color='lightyellow', edgecolor='orange')
axes[1, 0].set_title('F1-Score', fontweight='bold')
axes[1, 0].set_ylabel('F1-Score')
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(f1_scores):
    axes[1, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# AUC
aucs = test_results_df['auc'].values
axes[1, 1].bar(models_list, aucs, color='plum', edgecolor='purple')
axes[1, 1].set_title('ROC-AUC', fontweight='bold')
axes[1, 1].set_ylabel('ROC-AUC')
axes[1, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(aucs):
    axes[1, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# All metrics
x_pos = np.arange(len(models_list))
width = 0.15
axes[1, 2].bar(x_pos - 2*width, accuracies, width, label='Accuracy', alpha=0.8)
axes[1, 2].bar(x_pos - width, precisions, width, label='Precision', alpha=0.8)
axes[1, 2].bar(x_pos, recalls, width, label='Recall', alpha=0.8)
axes[1, 2].bar(x_pos + width, f1_scores, width, label='F1', alpha=0.8)
axes[1, 2].bar(x_pos + 2*width, aucs, width, label='AUC', alpha=0.8)
axes[1, 2].set_title('All Metrics Comparison', fontweight='bold')
axes[1, 2].set_ylabel('Score')
axes[1, 2].set_xticks(x_pos)
axes[1, 2].set_xticklabels(models_list, rotation=45)
axes[1, 2].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "06_all_models_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Comparison charts saved!")

print("\n" + "="*100)
print("✅ PROJECT COMPLETE - ALL RESULTS SAVED!")
print("="*100)
print(f"\n📁 Output files saved in:")
print(f"  Models: {MODELS_DIR}")
print(f"  Plots: {PLOTS_DIR}")
print(f"  Metrics: {METRICS_DIR}")